## Load Pdf Files

---

### 💡 Interview & Learning Notes

**Key Interview Questions:**
1. *Why are standalone integrations (like `langchain-pymupdf`) replacing `langchain-community`?* 
   - `langchain-community` became bloated and difficult to maintain. Standalone partner packages allow for independent versioning, tighter dependencies, and faster security updates without waiting for a core release.
2. *What are the common challenges when parsing PDFs for RAG?* 
   - PDFs are presentation formats, not structured data formats. Challenges include multi-column layouts, tables, headers/footers, ligatures (e.g., 'ﬁ'), and scanned images requiring OCR.
3. *How does PyMuPDF differ from PyPDF?* 
   - PyMuPDF is generally much faster and better at preserving layout and extracting images/annotations. PyPDF is simpler and purely python-based but can struggle with complex formatting.

**Learning Takeaways:**
- **Data Cleaning** is vital for PDFs. Always implement a cleaning function to remove ligatures and excessive whitespace before chunking.
- Use **enhanced metadata** (page numbers, chunk methods) to help trace the source of hallucinations later.

In [1]:
"""
PURPOSE:
Use modern standalone PDF loader packages.

INSIGHTS:
Standalone integrations are future stable.
"""

# Why are standalone LangChain integrations replacing langchain_community?
# Answer: To reduce bloat in the main repository and allow independent versioning of integrations.

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Component Explanations:
# 1. PyMuPDFLoader (from langchain_pymupdf):
#    - Purpose: Loads PDFs using the PyMuPDF library.
#    - Why Used: It's extremely fast and handles complex layouts, embedded links, and metadata better than pure python alternatives.
# 2. PyPDFLoader (from langchain_community.document_loaders):
#    - Purpose: Loads PDFs using the pypdf library.
#    - Why Used: Simple, widely supported, pure Python implementation. Good for simple text-based PDFs.

from langchain_community.document_loaders import PyMuPDFLoader

C:\Users\DELL\AppData\Local\Temp\ipykernel_21492\303867701.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [2]:
from langchain_community.document_loaders import PyPDFLoader
# (Note: PyPDFLoader remains in community for now, but always check for `langchain-pypdf` or similar in newer releases).

PDF_PATH = "data/pdf/attention.pdf"



In [3]:
# =========================
# PyPDFLoader
# =========================
print("1️⃣ PyPDFLoader")

try:
    pypdf_loader = PyPDFLoader(PDF_PATH)
    pypdf_docs = pypdf_loader.load()

    print(f"Loaded {len(pypdf_docs)} pages")
    print(f"Page 1 content:\n{pypdf_docs[0].page_content[:300]}...\n")
    print(f"Metadata:\n{pypdf_docs[0].metadata}")

except Exception as e:
    print(f"Error: {e}")



1️⃣ PyPDFLoader
Loaded 15 pages
Page 1 content:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...

Metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [4]:
# =========================
# PyMuPDFLoader
# =========================
print("\n2️⃣ PyMuPDFLoader")

try:
    pymupdf_loader = PyMuPDFLoader(PDF_PATH)
    pymupdf_docs = pymupdf_loader.load()

    print(f"Loaded {len(pymupdf_docs)} pages")
    print(f"Page 1 content:\n{pymupdf_docs[0].page_content[:300]}...\n")
    print(f"Metadata:\n{pymupdf_docs[0].metadata}")

except Exception as e:
    print(f"Error: {e}")


2️⃣ PyMuPDFLoader
Loaded 15 pages
Page 1 content:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...

Metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'data/pdf/attention.pdf', 'file_path': 'data/pdf/attention.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}


In [5]:
# =========================
# PDF Loader Comparison
# =========================
print("\n📊 PDF Loader Comparison:")

print("\n1️⃣ PyPDFLoader")
print("  ✅ Simple and reliable")
print("  ✅ Good for standard PDF text extraction")
print("  ✅ Preserves page-wise document structure")
print("  ✅ Stable for RAG pipelines")
print("  Use when: Working with normal text-based PDFs")

print("\n2️⃣ PyMuPDFLoader")
print("  ✅ Faster processing")
print("  ✅ Better text extraction accuracy")
print("  ✅ Handles complex PDF layouts better")
print("  ✅ Supports image extraction")
print("  Use when: Speed and extraction quality are important")


📊 PDF Loader Comparison:

1️⃣ PyPDFLoader
  ✅ Simple and reliable
  ✅ Good for standard PDF text extraction
  ✅ Preserves page-wise document structure
  ✅ Stable for RAG pipelines
  Use when: Working with normal text-based PDFs

2️⃣ PyMuPDFLoader
  ✅ Faster processing
  ✅ Better text extraction accuracy
  ✅ Handles complex PDF layouts better
  ✅ Supports image extraction
  Use when: Speed and extraction quality are important


### Handling PDF Challenges 
🎯 Purpose of This Section
PDFs are notoriously difficult to parse because they:

- Store text in complex ways (not just simple text)
- Can have formatting issues
- May contain scanned images (requiring OCR)
- Often have extraction artifacts


In [6]:
# =========================
# Raw PDF Text Cleaning
# =========================

raw_pdf_text = """
Company Financial Report


    The ﬁnancial performance for ﬁscal year 2024
    shows signiﬁcant growth in proﬁtability.



    Revenue increased by 25%.

The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10
"""


def clean_text(text: str) -> str:
    """
    Clean extracted PDF text for RAG pipelines.
    """

    # Fix common ligatures
    ligature_map = {
        "ﬁ": "fi",
        "ﬂ": "fl",
    }

    for wrong, correct in ligature_map.items():
        text = text.replace(wrong, correct)

    # Remove excessive whitespace/newlines
    text = " ".join(text.split())

    return text


# Apply cleaning
cleaned_text = clean_text(raw_pdf_text)

print("BEFORE:\n")
print(repr(raw_pdf_text))

print("\n" + "=" * 60)

print("\nAFTER:\n")
print(repr(cleaned_text))


# Output:
# BEFORE:
# 'Company Financial Report\n\n\n    The ﬁnancial performance for ﬁscal year 2024\n    shows signiﬁcan'
# 
# AFTER:
# 'Company Financial Report The financial performance for fiscal year 2024 shows significant growth in'

BEFORE:

"\nCompany Financial Report\n\n\n    The ﬁnancial performance for ﬁscal year 2024\n    shows signiﬁcant growth in proﬁtability.\n\n\n\n    Revenue increased by 25%.\n\nThe company's efﬁciency improved due to workﬂow\noptimization.\n\n\nPage 1 of 10\n"


AFTER:

"Company Financial Report The financial performance for fiscal year 2024 shows significant growth in profitability. Revenue increased by 25%. The company's efficiency improved due to workflow optimization. Page 1 of 10"


In [7]:
# Why do we load PDFs before splitting them into chunks for RAG?

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
"""
PURPOSE:
Process PDF files into chunks for RAG pipelines.

INSIGHTS:
Better chunking improves retrieval.
"""

from langchain_core.documents import Document
from typing import List

class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size
        self.chunk_overlap=chunk_overlap

        # Why do we split large PDF text into smaller chunks for RAG retrieval?
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n","\n","."," ",""],
        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Load PDF

        loader=PyPDFLoader(pdf_path)
        pages=loader.load()

        ## Process each page

        processed_chunks=[]

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks=self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            ) #return a list of chunks

            processed_chunks.extend(chunks)  

        return processed_chunks

    def _clean_text(self,text:str)->str:
        """Clean extracted text"""

        # Remove excessive whitespace
        text=" ".join(text.split())

        # Fix common PDF extraction issues
        text=text.replace("ﬁ","fi")
        text=text.replace("ﬂ","fl")

        return text

In [9]:
preprocessor=SmartPDFProcessor()

In [10]:
preprocessor

In [11]:
## Process a PDF if available
try:
    smart_chunks=preprocessor.process_pdf("data/pdf/attention.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")

        # Why is metadata important in RAG pipelines?
        # Answer: It allows for post-filtering, citations, and tracking the exact page a chunk came from.
        for key,value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

Processed into 49 smart chunks

Sample chunk metadata:
  producer: pdfTeX-1.40.25
  creator: LaTeX with hyperref
  creationdate: 2024-04-10T21:11:43+00:00
  author: 
  keywords: 
  moddate: 2024-04-10T21:11:43+00:00
  ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
  subject: 
  title: 
  trapped: /False
  source: data/pdf/attention.pdf
  total_pages: 15
  page: 1
  page_label: 1
  chunk_method: smart_pdf_processor
  char_count: 2857


### 🚀 Best Practices for PDF Ingestion

1. **Cost Efficiency & Token Optimization**:
   - Strip out repetitive headers, footers, and page numbers during the cleaning phase. These consume tokens without adding semantic value.
   - If a PDF has many images, decide whether to use a multimodal LLM (expensive) or to extract image captions/alt-text using PyMuPDF (cheaper).

2. **Time Optimization**:
   - Use `PyMuPDFLoader` instead of `PyPDFLoader` for large documents because of its C-based speed advantages.
   - Implement parallel processing (e.g., `ThreadPoolExecutor`) when loading multiple PDFs from a directory.

3. **Data Quality**:
   - Always run a `clean_text` step to fix ligatures (`ﬁ`, `ﬂ`) and arbitrary newlines that PDF extractors inject.
   - Append metadata such as `total_pages` and `page_num` for precise document citations in your UI.